# Setup

In [1]:
from pprint import pprint
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from transformers import set_seed
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Seed
seed = 42
set_seed(seed)

# Suppress loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1014 17:23:40.024000 19444 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1014 17:23:40.084000 19444 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1014 17:23:41.014000 19444 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for trito

# 1. Classification

> Note) Visit [here](../4_Model_Selection/1_Evaluation.ipynb) for implementation details.

## 1.1. Single Class

| Metric | Formula | API |
|---|---|---|
| Accuracy | $\mathrm{Accuracy}=\frac{TP+TN}{M}$ | `accuracy_score` |
| Precision | $P=\frac{TP}{TP+FP}$ | `precision_score` |
| Recall (Sensitivity) | $R=\frac{TP}{TP+FN}$ | `recall_score` |
| F1-score | $F_1=\frac{2PR}{P+R}=\frac{2TP}{2TP+FP+FN}$ | `f1_score` |
| ROC-AUC | $\mathrm{ROC\text{-}AUC}=\int_0^1 \mathrm{TPR}\,d(\mathrm{FPR})$<br>$\mathrm{TPR}=\frac{TP}{TP+FN}$<br>$\mathrm{FPR}=\frac{FP}{FP+TN}$ | `roc_auc_score` |
| PR-AUC | $\mathrm{PR\text{-}AUC}=\int_0^1 P(R)\,dR$ | `auc` |
| Average Precision (AP) | $\mathrm{AP}=\sum_t(R_t-R_{t-1})P_t$ | `average_precision_score` |
| Cross-Entropy | $\mathrm{CE}=-\frac{1}{M}\sum_{i=1}^{M}\sum_{k=1}^{K}y_{ik}\ln p_{ik}$ | `log_loss` |

## 1.2. Multi-Class

| Metric | Formula | API |
|---|---|---|
| Macro-F1 | $F_{1,\mathrm{macro}}=\frac{1}{K}\sum_{k=1}^{K}F_{1,k}$ | `f1_score`<br>`average="macro"` |
| Weighted-F1 | $F_{1,\mathrm{weighted}}=\sum_{k=1}^{K}\frac{M_k}{M}F_{1,k}$ | `f1_score`<br>`average="weighted"` |
| Micro-F1 | $F_{1,\mathrm{micro}}=\frac{2\sum_{k=1}^{K}TP_k}{2\sum_{k=1}^{K}TP_k+\sum_{k=1}^{K}FP_k+\sum_{k=1}^{K}FN_k}$ | `f1_score`<br>`average="micro"` |
| Balanced Accuracy | $\mathrm{BalancedAccuracy}=\frac{1}{K}\sum_{k=1}^{K}\frac{TP_k}{TP_k+FN_k}$ | `balanced_accuracy_score` |

# 2. Regression

> Note) Visit [here](../4_Model_Selection/1_Evaluation.ipynb) for implementation details.

| Metric | Formula | scikit-learn |
|---|---|---|
| MAE | $\mathrm{MAE}=\frac{1}{M}\sum_{i=1}^{M}\lvert y_i-\hat{y}_i\rvert$ | `mean_absolute_error` |
| MSE | $\mathrm{MSE}=\frac{1}{M}\sum_{i=1}^{M}(y_i-\hat{y}_i)^2$ | `mean_squared_error` |
| RMSE | $\mathrm{RMSE}=\sqrt{\frac{1}{M}\sum_{i=1}^{M}(y_i-\hat{y}_i)^2}=\sqrt{\mathrm{MSE}}$ | `root_mean_squared_error` |
| $R^2$ (Coefficient of Determination) | $R^2=1-\frac{\sum_{i=1}^{M}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{M}(y_i-\bar{y})^2}$<br>$\bar{y}=\frac{1}{M}\sum_{i=1}^{M}y_i$ | `r2_score` |
| MAPE | $\mathrm{MAPE}=\frac{1}{M}\sum_{i=1}^{M}\left\lvert\frac{y_i-\hat{y}_i}{y_i}\right\rvert$ | `mean_absolute_percentage_error` |

# 3. NLP Evaluation

## 3.1. BLEU

- Definition
  - Bilingual Evaluation Understudy
  - How many generated n-grams appear in the reference?
    - **Generation -> Reference**
  - $BP = \begin{cases} 1 &  (c \geq r) \\ e^{1-r/c} &  (c < r) \end{cases}$
    - Brevity penalty: penalizes excessively short predictions.
    - $c$: Length of generated text.
    - $r$: Length of reference text.
    - $e$: Euler's number ($\approx 2.718$).
  - $BLEU\text{-}N = BP \times \left(\prod_{n=1}^{N} p_n\right)^{1/N}$
    - $N$: Maximum n-gram order (typically 4).
    - $n$: Current n-gram order (1 = unigram, 2 = bigram, etc.).
    - $p_n$: Modified precision for n-grams of order $n$.
    - $\prod$: Product operator (multiplies all $p_n$ from $n=1$ to $N$).
    - $1/N$: Computes the geometric mean of n-gram precisions.
- Main Purpose
  - Machine translation.
- Limitation
  - Could not catch a semantic similarity.

In [2]:
import evaluate

predictions = ["the cat is on the mat"]
references = ["the cat is on a mat"]

bleu = evaluate.load('bleu')

bleu_result = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

pprint(bleu_result)     # 'precisions': 1-4 gram precision

{'bleu': 0.537284965911771,
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'precisions': [0.8333333333333334, 0.6, 0.5, 0.3333333333333333],
 'reference_length': 6,
 'translation_length': 6}


## 3.2. ROUGE

- ROUGE
  - Recall-Oriented Understudy for Gisting Evaluation
  - How many reference n-grams appear in the generated text?
    - **Reference → Generation**
  - $ROUGE\text{-}N = \frac{\text{Matched reference n-grams}}{\text{Total reference n-grams}}$
    - $N$: N-gram order (1 = unigram, 2 = bigram, etc.).
    - Matched n-grams: Overlapping n-grams between reference and generation (with clipped counts).
  - Types
    - ROUGE-1: Unigram overlap.
    - ROUGE-2: Bigram overlap.
    - ROUGE-L: Longest common subsequence (LCS).
      - Allows gaps, but word order must be preserved.
      - $ROUGE\text{-}L_{\text{recall}} = \frac{LCS(gen, ref)}{\text{Length of reference}}$
- Main Purpose
  - Text summarization.
- Limitation
  - Cannot capture semantic similarity.

> Note) `evaluate.load('rouge')` returns **f1**, not **recall**.

In [3]:
import evaluate

predictions = ["the cat is on the mat"]
references = ["the cat is on a mat"]

rouge = evaluate.load('rouge')

rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

pprint(rouge_result)

{'rouge1': np.float64(0.8333333333333334),
 'rouge2': np.float64(0.6),
 'rougeL': np.float64(0.8333333333333334),
 'rougeLsum': np.float64(0.8333333333333334)}


## 3.3. Semantic Evaluation

- Compares the similarity of embeddings.
- Limitation
  - could catch a meaning closeness, though not factual correctness.
  - e.g. "It is safe" vs "It is not safe".

### Sentence + Cos

- $score = cos(e_{pred}, e_{ref})$

In [4]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

pred = "It is not safe."
ref = "It is safe."

e_pred = model.encode(pred)
e_ref = model.encode(ref)

score = util.cos_sim(e_pred, e_ref).item()
print(score)

0.9061764478683472


### Token Embeddings

In [5]:
import evaluate

bertscore = evaluate.load('bertscore')

result = bertscore.compute(
    predictions=["It is not safe."],
    references=["It is safe."],
    lang="en"
)

print(result)

{'precision': [0.9635886549949646], 'recall': [0.9779958724975586], 'f1': [0.9707387685775757], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.18.0)'}


## 3.4. Pass@k

- pass@1 = probability that one generated solution is correct.
- pass@k = probability that at least one of k generated solutions is correct.

## 3.5. best-of-N

- Generate N answers, then a scorer/reward model selects one.

## 3.6. LLM-as-a-Judge

```markdown
Question:
What is the capital of France?

Response:
Paris is the capital of France.

Judge criteria:
- factual correctness
- relevance
- clarity

LLM answer:
correctness: 4/5
relevance:   5/5
clarity:     4/5
```

- Approaches
  - Pointwise: score one response independently
  - Pairwise: compare response A vs response B
- Limitations
  - Position bias: the ordering of A/B affects the scoring
  - Verbosity bias: prefer longer answers
  - Self-preference: prefer outputs similar to the judge model's own style
  - Prompt sensitivity: small changes could make a huge difference on scoring

# 4. Benchmarks

| Benchmark                                                     | Tests                                                                    | Typical metric             | Current frontier snapshot                                                                                                     | Status                                                  |
| ------------------------------------------------------------- | ------------------------------------------------------------------------ | -------------------------- | ----------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------- |
| **MMLU** — Massive Multitask Language Understanding           | Broad academic knowledge across many subjects                            | Accuracy                   | Frontier models are well above 90%                                                                                            | **Legacy / largely saturated**                          |
| **MMLU-Pro**                                                  | Harder knowledge + reasoning, 10-choice questions across 14 domains      | Accuracy                   | Gemini 3 Pro Preview (high): **89.8%** ([Artificial Analysis][1])                                                             | Still useful, but becoming mature                       |
| **GPQA Diamond** — Graduate-Level Google-Proof Q&A            | PhD-level physics, chemistry, biology reasoning                          | Accuracy                   | GPT-6 Astra (xhigh): **96.3%** ([Artificial Analysis][2])                                                                     | Frontier science reasoning, increasingly saturated      |
| **AIME 2025** — American Invitational Mathematics Examination | Competition mathematics                                                  | Exact-answer accuracy      | Multiple models at **100%**, including GPT-5.2 (xhigh) ([Artificial Analysis][3])                                             | Already saturated at frontier                           |
| **MATH-500**                                                  | Competition-level mathematics                                            | Accuracy                   | GPT-5 (high): **99.4%** ([Artificial Analysis][4])                                                                            | Legacy / saturated                                      |
| **LiveCodeBench**                                             | Fresh competitive-programming problems; designed to reduce contamination | Pass/correctness score     | Gemini 3 Pro Preview (high): **91.7%** ([Artificial Analysis][5])                                                             | Much more useful than old HumanEval for frontier coding |
| **SWE-bench Verified**                                        | Fix real GitHub issues in real repositories                              | `% resolved`               | Claude 4.5 Opus + agent harness: **79.2%** ([SWE-bench][6])                                                                   | Very important for coding agents                        |
| **IFBench**                                                   | Precise instruction following under verifiable constraints               | Constraint-following score | Grok 4.3 (medium): **83.3%** ([Artificial Analysis][7])                                                                       | Useful modern instruction-following benchmark           |
| **HLE** — Humanity's Last Exam                                | Expert-level math, science, humanities                                   | pass@1                     | Claude Fable 5.1, max effort: **59.1%** ([Artificial Analysis][8])                                                            | Current frontier academic benchmark                     |
| **LM Arena**                                                  | Human preference on open-ended model responses                           | Pairwise Elo-style rating  | Extremely dynamic and category-dependent; e.g. Software & IT currently has Claude Opus 4.7-high around **1541** ([아레나 AI][9]) | Important for subjective/chat quality                   |

[1]: https://artificialanalysis.ai/evaluations/mmlu-pro?utm_source=chatgpt.com "MMLU-Pro Benchmark Leaderboard | Artificial Analysis"
[2]: https://artificialanalysis.ai/evaluations/gpqa-diamond?utm_source=chatgpt.com "GPQA Diamond Benchmark Leaderboard | Artificial Analysis"
[3]: https://artificialanalysis.ai/evaluations/aime-2025?models=gpt-4-1%2Cgpt-oss-120b%2Cgpt-oss-20b%2Cgpt-5%2Co3%2Cgpt-5-minimal%2Cllama-4-maverick%2Cgemini-2-5-pro%2Cgemini-2-5-flash-reasoning%2Cclaude-4-1-opus-thinking%2Cclaude-4-sonnet-thinking%2Cmistral-medium-3-1%2Cdeepseek-r1%2Cdeepseek-v3-1-reasoning%2Cdeepseek-v3-1%2Cgrok-code-fast-1%2Cgrok-4%2Csolar-pro-2-reasoning%2Cllama-nemotron-super-49b-v1-5-reasoning%2Ckimi-k2-0905%2Cexaone-4-0-32b-reasoning%2Cglm-4.5%2Cqwen3-235b-a22b-instruct-2507-reasoning&utm_source=chatgpt.com "AIME 2025 Benchmark Leaderboard | Artificial Analysis"
[4]: https://artificialanalysis.ai/evaluations/math-500?utm_source=chatgpt.com "MATH-500 Benchmark Leaderboard | Artificial Analysis"
[5]: https://artificialanalysis.ai/evaluations/livecodebench?utm_source=chatgpt.com "LiveCodeBench Benchmark Leaderboard | Artificial Analysis"
[6]: https://www.swebench.com/?sf206742404=1&utm_source=chatgpt.com "SWE-bench Leaderboards"
[7]: https://artificialanalysis.ai/evaluations/ifbench "IFBench Benchmark Leaderboard | Artificial Analysis"
[8]: https://artificialanalysis.ai/evaluations/humanitys-last-exam "Humanity's Last Exam Benchmark Leaderboard | Artificial Analysis"
[9]: https://lmarena.ai/leaderboard/text/industry-software-and-it-services?utm_source=chatgpt.com "LLM Leaderboard - Best Text & Chat AI Models Compared"


- Order benchmarks
  - GSM8K      → grade-school math reasoning
  - HumanEval  → Python code generation
  - MBPP       → basic Python programming
  - ARC        → science QA / reasoning
  - HellaSwag  → commonsense completion
  - TruthfulQA → truthfulness / misconceptions
  - MMLU       → broad academic knowledge

 > Note) Benchmark decontamination
 > - A large LLM could have already seen a benchmark dataset in pretraining corpus.
 > - No specific universe answer, though must be aware.



# 5. Evaluation Variance

- Training randomness
  - Initialization
  - Data order
  - Dropout
- Generation randomness
  - Sampling with temperature / top-p / top-k
- Ablation study
  - Different seeds -> reproducibility, consistent improvements.
  - Mean/std of confidence intervals.

# 6. Evaluation Harnesses

## 6.1. APIs

- lm-evaluation-harness
  - EleutherAI
  - Classic/common benchmark runner
  - Supports many zero-shot/few-shot tasks
- Lighteval
  - Hugging Face
  - Modern multi-backend evaluation toolkit
  - Supports transformers, vLLM, SGLang, APIs, and custom tasks.

## 6.2. Example - Qwen

- Task: HellaSwag
- Model: `Qwen2.5-0.5B`
- Harness: EleutherAI (`lm_eval`)

In [6]:
import lm_eval

results = lm_eval.simple_evaluate(
    model="hf",
    model_args={"pretrained": "Qwen/Qwen2.5-0.5B"},
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=20,
    device="cuda",
    batch_size=4,
)

print(results["results"]["hellaswag"])

Running loglikelihood requests: 100%|██████████| 80/80 [00:01<00:00, 55.24it/s]


{'name': 'hellaswag', 'alias': 'hellaswag', 'sample_len': 20, 'acc,none': 0.3, 'acc_stderr,none': 0.10513149660756935, 'acc_norm,none': 0.35, 'acc_norm_stderr,none': 0.1094243309804831}


## 6.3. Example - Custom Model

In [7]:
import lm_eval
from lm_eval.api.model import LM

class MyLM(LM):
    def __init__(self):
        super().__init__()

    def loglikelihood(self, requests):
        results = []

        for request in requests:
            context, continuation = request.args

            score = -len(continuation)  # dummy score: shorter continuations are more likely.

            results.append((score, False))

        return results

    def loglikelihood_rolling(self, requests):
        # Used for perplexity-type tasks.
        return [0.0 for _ in requests]

    def generate_until(self, requests):
        # Used for generation-type tasks.
        return ["" for _ in requests]


model = MyLM()

results = lm_eval.simple_evaluate(
    model=model,
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=10,
)

print(results["results"]["hellaswag"])

100%|██████████| 10/10 [00:00<00:00, 1498.93it/s]


{'name': 'hellaswag', 'alias': 'hellaswag', 'sample_len': 10, 'acc,none': 0.3, 'acc_stderr,none': 0.15275252316519464, 'acc_norm,none': 0.4, 'acc_norm_stderr,none': 0.16329931618554522}
